> **Version del mentor, con soluciones.** No compartir antes de la clase.

# Clase guiada: limpieza y estructura de datos de RRHH
**Dataset:** IBM HR Employee Attrition (1.470 empleados, 35 columnas) — version "ensuciada" a proposito.
**Duracion:** 40 minutos · **Herramientas:** Python + pandas

**Pregunta de negocio que guia todo:** *¿que caracteristicas tienen las personas que se van de la empresa?*
Hoy NO respondemos esa pregunta. Hoy dejamos los datos en condiciones de poder responderla.

| Bloque | Minutos | Que hacemos |
|---|---|---|
| 0. Contexto | 0–5 | Por que limpiar y como vamos a trabajar |
| 1. Cargar y mirar | 5–10 | Primer contacto con los datos |
| 2. Diagnosticar | 10–17 | Encontrar los problemas **antes** de tocar nada |
| 3. Limpiar | 17–30 | Duplicados, texto, tipos, valores imposibles, nulos |
| 4. Estructurar | 30–35 | Escalas legibles y variables nuevas |
| 5. Validar y guardar | 35–40 | Chequeos automaticos + bitacora de decisiones |

> Las celdas marcadas con ✏️ son **para vos**. Las demas las vemos juntos.

## 0. Tres reglas de trabajo
1. **Nunca pisamos el original.** Trabajamos sobre una copia; el archivo crudo queda intacto.
2. **Primero diagnostico, despues limpieza.** Si arreglas lo primero que ves, te perdes lo demas.
3. **Toda decision queda escrita.** "Imputé el sueldo con la mediana del nivel" es una decision de negocio, no solo de codigo. Alguien te la va a preguntar.

## 1. Cargar y mirar (min 5–10)

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

# Si todavia no generaste el archivo sucio:
#   python ensuciar_datos.py WA_Fn-UseC_-HR-Employee-Attrition.csv
crudo = pd.read_csv("HR_sucio.csv", encoding="utf-8-sig")  # utf-8-sig: el CSV original trae un caracter invisible (BOM) al inicio
crudo.shape

(1499, 35)

In [2]:
crudo.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,55,No,Travel_Rarely,685,Sales,26.0,5,Marketing,1,1578,3,Male,60,2,5,Manager,4,Married,19586,23037,1,Y,No,21,4,3,80,1,36,3,3,36,6,2,13
1,31,Yes,Travel_Frequently,754,Sales,26.0,4,Marketing,1,1967,1,Male,63,3,2,Sales Executive,4,Married,5617,21075,1,Y,Yes,11,3,3,80,0,10,4,3,10,7,0,8
2,19,Yes,Travel_Rarely,489,Human Resources,2.0,2,Technical Degree,1,566,1,Male,52,2,1,Human Resources,4,Single,2564,18437,1,Y,No,12,3,3,80,0,1,3,4,1,0,0,0
3,42,No,Travel_Frequently,1368,Research & Development,28.0,4,Technical Degree,1,551,4,Female,88,2,2,Healthcare Representative,4,Married,4523,4386,0,Y,No,11,3,4,80,3,7,4,4,6,5,0,4
4,42,No,Travel_Rarely,319,Research & Development,24.0,3,Medical,1,605,4,Male,56,3,3,Manufacturing Director,1,Married,7406,6950,1,Y,Yes,21,4,4,80,1,10,5,2,10,9,5,8


In [3]:
crudo.info()

<class 'pandas.DataFrame'>
RangeIndex: 1499 entries, 0 to 1498
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1499 non-null   int64  
 1   Attrition                 1499 non-null   str    
 2   BusinessTravel            1499 non-null   str    
 3   DailyRate                 1499 non-null   int64  
 4   Department                1484 non-null   str    
 5   DistanceFromHome          1468 non-null   float64
 6   Education                 1499 non-null   int64  
 7   EducationField            1499 non-null   str    
 8   EmployeeCount             1499 non-null   int64  
 9   EmployeeNumber            1499 non-null   int64  
 10  EnvironmentSatisfaction   1499 non-null   int64  
 11  Gender                    1499 non-null   str    
 12  HourlyRate                1499 non-null   int64  
 13  JobInvolvement            1499 non-null   int64  
 14  JobLevel           

**Pregunta para charlar:** mirando `info()`, ¿hay alguna columna cuyo *tipo* no tenga sentido?
<details><summary>Pista</summary>¿Un sueldo deberia ser texto?</details>

## 2. Diagnosticar (min 10–17)
Objetivo: una lista de problemas. **Todavia no arreglamos nada.**

In [4]:
# Nulos por columna (solo las que tienen alguno)
nulos = crudo.isna().sum()
nulos[nulos > 0]

Department          15
DistanceFromHome    31
MonthlyIncome       45
dtype: int64

In [5]:
# a) Filas duplicadas
print("Duplicados exactos:", crudo.duplicated().sum())

# b) Columnas constantes: no aportan informacion (todos tienen el mismo valor)
constantes = crudo.columns[crudo.nunique() == 1].tolist()
constantes

Duplicados exactos: 29


['EmployeeCount', 'Over18', 'StandardHours']

In [6]:
columnas_texto = crudo.select_dtypes(exclude="number").columns
for col in columnas_texto:
    if crudo[col].nunique() < 15:  # las de pocas categorias; MonthlyIncome la vemos aparte
        print(crudo[col].value_counts(dropna=False), "\n")

Attrition
No     1212
Yes     229
N        20
NO       16
no        8
Si        8
Y         4
yes       2
Name: count, dtype: int64 

BusinessTravel
Travel_Rarely        1062
Travel_Frequently     284
Non-Travel            153
Name: count, dtype: int64 

Department
Research & Development     870
Sales                      407
Human Resources             59
R&D                         44
 Research & Development     36
research & development      20
Sales                       16
NaN                         15
sales                       14
SALES                       14
HR                           3
human resources              1
Name: count, dtype: int64 

EducationField
Life Sciences       617
Medical             475
Marketing           164
Technical Degree    134
Other                82
Human Resources      27
Name: count, dtype: int64 

Gender
Male      855
Female    556
male       17
MALE       15
female     15
F          15
Fem        14
M          12
Name: count, dtype: int64 



In [7]:
# Rangos de las numericas: ¿algo imposible?
crudo.describe().T[["min", "max"]]

,min,max
Age,-1.0,999.0
DailyRate,102.0,1499.0
DistanceFromHome,1.0,29.0
Education,1.0,5.0
EmployeeCount,1.0,1.0
EmployeeNumber,1.0,2068.0
EnvironmentSatisfaction,1.0,4.0
HourlyRate,30.0,100.0
JobInvolvement,1.0,4.0
JobLevel,1.0,5.0


**Lista de problemas que deberiamos tener a esta altura** (completar juntos antes de seguir):
- [ ] Filas duplicadas
- [ ] Columnas constantes (`EmployeeCount`, `Over18`, `StandardHours`)
- [ ] `Department`, `Gender`, `Attrition`, `OverTime` escritos de muchas formas
- [ ] `MonthlyIncome` guardado como texto (`"$5,993"`)
- [ ] Edades imposibles (`Age` = -1, 4, 150, 999)
- [ ] Nulos en `MonthlyIncome`, `DistanceFromHome`, `Department`

## 3. Limpiar (min 17–30)
Trabajamos sobre `df` (copia) y registramos cada decision en `bitacora`.

In [8]:
df = crudo.copy()
bitacora = []

def registrar(decision):
    bitacora.append(decision)
    print("📝", decision)

### 3.1 Duplicados

In [9]:
antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
registrar(f"Elimine {antes - len(df)} filas duplicadas exactas")

📝 Elimine 29 filas duplicadas exactas


> **Para pensar:** en un sistema real, ¿alcanza con buscar filas *identicas*? ¿Que pasa si el mismo `EmployeeNumber` aparece dos veces con datos distintos?

In [10]:
df = df.drop(columns=constantes)
registrar(f"Elimine columnas constantes: {constantes}")

📝 Elimine columnas constantes: ['EmployeeCount', 'Over18', 'StandardHours']


### 3.2 Texto inconsistente
Receta: 1) sacar espacios, 2) llevar a un formato comun, 3) mapear sinonimos con un diccionario.

In [11]:
df["Department"] = df["Department"].str.strip().str.lower().replace({
    "sales": "Sales",
    "research & development": "Research & Development",
    "r&d": "Research & Development",
    "human resources": "Human Resources",
    "hr": "Human Resources",
})
registrar("Department: unifiqué mayúsculas, espacios y abreviaturas (R&D, HR)")
df["Department"].value_counts(dropna=False)

📝 Department: unifiqué mayúsculas, espacios y abreviaturas (R&D, HR)


Department
Research & Development    952
Sales                     441
Human Resources            63
NaN                        14
Name: count, dtype: int64

In [12]:
df["Gender"] = df["Gender"].str.strip().str.lower().replace({
    "m": "Male", "male": "Male",
    "f": "Female", "fem": "Female", "female": "Female",
})

a_booleano = {"yes": True, "y": True, "si": True, "no": False, "n": False}
for col in ["Attrition", "OverTime"]:
    df[col] = df[col].str.strip().str.lower().map(a_booleano)
    assert df[col].notna().all(), f"Quedaron valores sin mapear en {col}"

registrar("Unifique Gender en Male/Female y pase Attrition/OverTime a booleano")
for col in ["Gender", "Attrition", "OverTime"]:
    print(df[col].value_counts(), "\n")

📝 Unifique Gender en Male/Female y pase Attrition/OverTime a booleano
Gender
Male      882
Female    588
Name: count, dtype: int64 

Attrition
False    1233
True      237
Name: count, dtype: int64 

OverTime
False    1054
True      416
Name: count, dtype: int64 



### 3.3 Numeros guardados como texto

In [13]:
sueldo_texto = df["MonthlyIncome"].astype("string").str.replace(r"[$,]", "", regex=True)
df["MonthlyIncome"] = pd.to_numeric(sueldo_texto, errors="coerce")
registrar("MonthlyIncome: saque '$' y ',' y lo converti a numero")
df["MonthlyIncome"].describe()

📝 MonthlyIncome: saque '$' y ',' y lo converti a numero


count         1426.0
mean     6506.621318
std      4717.366424
min           1009.0
25%           2909.5
50%           4936.0
75%           8379.0
max          19999.0
Name: MonthlyIncome, dtype: Float64

### 3.4 Valores imposibles
**Decision de negocio:** ¿borramos la fila entera o solo el dato malo?
Si borramos la fila perdemos el resto de la informacion de esa persona (que esta bien). Preferimos marcar el dato como faltante.

In [14]:
edad_min, edad_max = 18, 70
fuera_de_rango = ~df["Age"].between(edad_min, edad_max)
df.loc[fuera_de_rango, "Age"] = np.nan
registrar(f"Age: {fuera_de_rango.sum()} valores fuera de [{edad_min}, {edad_max}] pasados a NaN")

📝 Age: 4 valores fuera de [18, 70] pasados a NaN


### 3.5 Nulos: cada columna merece su propia decision
| Columna | Idea | Por que |
|---|---|---|
| `MonthlyIncome` | Mediana **segun `JobLevel`** | El sueldo depende mucho del nivel; la mediana general seria injusta para un gerente |
| `Department` | Deducirlo de `JobRole` | Casi todos los puestos pertenecen a un solo departamento |
| `Age`, `DistanceFromHome` | Mediana general | Pocos casos, sin una variable que los explique mejor |

Buena practica: **dejar una marca** de que el valor fue imputado.

In [15]:
df["sueldo_imputado"] = df["MonthlyIncome"].isna()
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(
    df.groupby("JobLevel")["MonthlyIncome"].transform("median")
)
registrar(f"MonthlyIncome: imputé {df['sueldo_imputado'].sum()} nulos con la mediana de su JobLevel")

📝 MonthlyIncome: imputé 44 nulos con la mediana de su JobLevel


In [16]:
# a) Puestos que pertenecen a un unico departamento
tabla = pd.crosstab(df["JobRole"], df["Department"])
depto_por_puesto = tabla[(tabla > 0).sum(axis=1) == 1].idxmax(axis=1)
print(depto_por_puesto, "\n")

# b) Completar con el departamento del puesto; los Manager quedan como "Desconocido"
nulos_depto = df["Department"].isna().sum()
df["Department"] = df["Department"].fillna(df["JobRole"].map(depto_por_puesto)).fillna("Desconocido")
registrar(f"Department: completé {nulos_depto} nulos a partir de JobRole ('Desconocido' si era Manager)")

# c) Mediana para el resto
for col in ["Age", "DistanceFromHome"]:
    n_nulos = df[col].isna().sum()
    df[col] = df[col].fillna(df[col].median())
    registrar(f"{col}: imputé {n_nulos} nulos con la mediana")

df.isna().sum().sum()

JobRole
Healthcare Representative    Research & Development
Human Resources                     Human Resources
Laboratory Technician        Research & Development
Manufacturing Director       Research & Development
Research Director            Research & Development
Research Scientist           Research & Development
Sales Executive                               Sales
Sales Representative                          Sales
dtype: str 

📝 Department: completé 14 nulos a partir de JobRole ('Desconocido' si era Manager)
📝 Age: imputé 4 nulos con la mediana
📝 DistanceFromHome: imputé 29 nulos con la mediana


np.int64(0)

## 4. Estructurar (min 30–35)
Muchas columnas son escalas codificadas con numeros (1 a 4). Para leer y graficar conviene tener la etiqueta,
y para el modelo conviene que pandas sepa que **hay un orden** (Low < Medium < High).

In [17]:
escalas = {
    "Education": {1: "Below College", 2: "College", 3: "Bachelor", 4: "Master", 5: "Doctor"},
    "EnvironmentSatisfaction": {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "JobSatisfaction":         {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "RelationshipSatisfaction":{1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "JobInvolvement":          {1: "Low", 2: "Medium", 3: "High", 4: "Very High"},
    "WorkLifeBalance":         {1: "Bad", 2: "Good", 3: "Better", 4: "Best"},
    "PerformanceRating":       {1: "Low", 2: "Good", 3: "Excellent", 4: "Outstanding"},
}
for col, etiquetas in escalas.items():
    orden = list(etiquetas.values())
    df[col + "_txt"] = pd.Categorical(df[col].map(etiquetas), categories=orden, ordered=True)

registrar("Agregué versiones con etiqueta (ordenadas) de las escalas 1-4/1-5; mantuve el número original")
df[["JobSatisfaction", "JobSatisfaction_txt"]].head()

📝 Agregué versiones con etiqueta (ordenadas) de las escalas 1-4/1-5; mantuve el número original


,JobSatisfaction,JobSatisfaction_txt
0,4,Very High
1,4,Very High
2,4,Very High
3,4,Very High
4,1,Low


In [18]:
df["tramo_edad"] = pd.cut(df["Age"], bins=[17, 25, 35, 45, 55, 70],
                          labels=["18-25", "26-35", "36-45", "46-55", "56+"])
df["tramo_antiguedad"] = pd.cut(df["YearsAtCompany"], bins=[-1, 0, 2, 5, 10, 100],
                                labels=["<1", "1-2", "3-5", "6-10", "+10"])
# Proporcion de la antigüedad que pasó sin ascenso (0 = recien ascendido, 1 = nunca ascendio)
df["prop_sin_ascenso"] = (df["YearsSinceLastPromotion"] / df["YearsAtCompany"].replace(0, np.nan)).fillna(0)
registrar("Agregué tramo_edad, tramo_antiguedad y prop_sin_ascenso")
df[["tramo_edad", "tramo_antiguedad", "prop_sin_ascenso"]].describe(include="all")

📝 Agregué tramo_edad, tramo_antiguedad y prop_sin_ascenso


,tramo_edad,tramo_antiguedad,prop_sin_ascenso
count,1470,1470,1470.000000
unique,5,5,NaN
top,26-35,6-10,NaN
freq,605,448,NaN
mean,NaN,NaN,0.290232
std,NaN,NaN,0.340521
min,NaN,NaN,0.000000
25%,NaN,NaN,0.000000
50%,NaN,NaN,0.166667
75%,NaN,NaN,0.500000


> **Para pensar (sin codigo):** `DailyRate`, `HourlyRate` y `MonthlyRate` casi no se correlacionan con `MonthlyIncome`.
> ¿Que significan? Si no podes explicar una columna, anotalo como pregunta abierta antes de usarla en un modelo.
>
> **Etica:** `Gender`, `Age` y `MaritalStatus` son datos sensibles. Sirven para *detectar* inequidades,
> pero hay que pensar dos veces antes de usarlos para *decidir* sobre personas.

## 5. Validar y guardar (min 35–40)
Los chequeos se escriben **una vez** y se corren **cada vez** que cambia el proceso.
Si mañana llega un archivo nuevo, te avisan solos si algo se rompio.

In [19]:
def chequear(condicion, mensaje):
    print(("OK   " if condicion else "FALLA") + " | " + mensaje)

chequear(not df.duplicated().any(), "sin filas duplicadas")
chequear(df["EmployeeNumber"].is_unique, "un registro por empleado")
chequear(df.drop(columns=[c + "_txt" for c in escalas]).notna().all().all(), "sin nulos (fuera de las etiquetas)")
chequear(df["Age"].between(18, 70).all(), "edades en rango")
chequear(df["Gender"].isin(["Male", "Female"]).all(), "Gender con 2 valores")
chequear(pd.api.types.is_numeric_dtype(df["MonthlyIncome"]), "MonthlyIncome es numerico")

OK    | sin filas duplicadas
OK    | un registro por empleado
OK    | sin nulos (fuera de las etiquetas)
OK    | edades en rango
OK    | Gender con 2 valores
OK    | MonthlyIncome es numerico


In [20]:
chequear((df["YearsAtCompany"] <= df["TotalWorkingYears"]).all(), "antigüedad <= años de experiencia")
tasa = df["Attrition"].mean()
chequear(0.14 < tasa < 0.18, f"tasa de rotacion razonable ({tasa:.1%})")

OK    | antigüedad <= años de experiencia
OK    | tasa de rotacion razonable (16.1%)


In [21]:
df.to_csv("HR_limpio.csv", index=False)
pd.Series(bitacora, name="decision").to_csv("bitacora_limpieza.csv", index=False)
print(f"Guardado: {df.shape[0]} filas x {df.shape[1]} columnas\n")
print("\n".join(f"{i}. {d}" for i, d in enumerate(bitacora, 1)))

Guardado: 1470 filas x 43 columnas

1. Elimine 29 filas duplicadas exactas
2. Elimine columnas constantes: ['EmployeeCount', 'Over18', 'StandardHours']
3. Department: unifiqué mayúsculas, espacios y abreviaturas (R&D, HR)
4. Unifique Gender en Male/Female y pase Attrition/OverTime a booleano
5. MonthlyIncome: saque '$' y ',' y lo converti a numero
6. Age: 4 valores fuera de [18, 70] pasados a NaN
7. MonthlyIncome: imputé 44 nulos con la mediana de su JobLevel
8. Department: completé 14 nulos a partir de JobRole ('Desconocido' si era Manager)
9. Age: imputé 4 nulos con la mediana
10. DistanceFromHome: imputé 29 nulos con la mediana
11. Agregué versiones con etiqueta (ordenadas) de las escalas 1-4/1-5; mantuve el número original
12. Agregué tramo_edad, tramo_antiguedad y prop_sin_ascenso


## Cierre
**Tres preguntas para vos:**
1. ¿Cual de las decisiones de hoy te parece mas discutible? ¿Como la defenderias frente a un gerente?
2. Si el dataset tuviera 100 veces mas filas, ¿que cambiarias del proceso?
3. ¿Que columna **falta** para responder "por que se va la gente"? (pensalo desde RRHH, no desde Python)

**Para la proxima (lo haces vos):**
- Subir `HR_limpio.csv` a MySQL (pista: `sqlalchemy.create_engine` + `df.to_sql`) y repetir 2 chequeos en SQL.
- Primer analisis exploratorio: tasa de rotacion por `Department`, `OverTime`, `tramo_edad` y `JobSatisfaction_txt`.
- Traer 3 hallazgos y 1 pregunta que no hayas podido responder.